In [3]:
from pathlib import Path
from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

all_pages = []

for file in Path(RAW_DATA_DIR).glob("*.txt"):
    pages = load_text(str(file))
    all_pages.extend(pages)

chunks = build_chunks(
    document_id="multi_doc",
    title="Combined Policies",
    pages=all_pages
)

print("\nTotal chunks:", len(chunks))

for c in chunks:
    print("-" * 50)
    print("clause_id:", c.clause_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text)


Total chunks: 6
--------------------------------------------------
clause_id: 1_0
section_id: 1
section_title: Access Control
text: All employees must authenticate using multi-factor authentication.
--------------------------------------------------
clause_id: 1_1
section_id: 2
section_title: Privileged Access
text: Administrative access must be reviewed every 30 days.
--------------------------------------------------
clause_id: 1_2
section_id: 3
section_title: Violations
text: Unauthorized access attempts may result in account suspension.
--------------------------------------------------
clause_id: 1_0
section_id: 1
section_title: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.
--------------------------------------------------
clause_id: 1_1
section_id: 2
section_title: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
--------------

In [4]:
import json
from pathlib import Path

from lexguard.utils.paths import PROJECT_ROOT
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

benchmark_path = PROJECT_ROOT / "data" / "benchmarks" / "retrieval_smoke_test.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

bm25 = BM25Retriever(chunks)
dense = DenseRetriever(chunks)
hybrid = HybridRetriever(chunks, bm25_weight=0.5, dense_weight=0.5)

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
}

results = []

for item in benchmark:
    query = item["query"]
    expected = item["expected_section"]

    row = {
        "query": query,
        "expected": expected,
    }

    for name, system in systems.items():
        pred = system.query(query, top_k=1)[0].section_title
        row[name] = pred
        row[f"{name}_hit"] = int(pred == expected)

    results.append(row)

for row in results:
    print("=" * 80)
    print("QUERY:", row["query"])
    print("EXPECTED:", row["expected"])
    print("BM25   :", row["bm25"], "| hit =", row["bm25_hit"])
    print("DENSE  :", row["dense"], "| hit =", row["dense_hit"])
    print("HYBRID :", row["hybrid"], "| hit =", row["hybrid_hit"])

print("\nSummary")
for name in systems:
    score = sum(r[f"{name}_hit"] for r in results) / len(results)
    print(f"{name}: hit@1 = {score:.2f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3708.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3739.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: log retention period
EXPECTED: Data Retention
BM25   : Data Retention | hit = 1
DENSE  : Exceptions | hit = 0
HYBRID : Data Retention | hit = 1
QUERY: financial logs retention
EXPECTED: Exceptions
BM25   : Exceptions | hit = 1
DENSE  : Exceptions | hit = 1
HYBRID : Exceptions | hit = 1
QUERY: temporary logs deletion
EXPECTED: Deletion
BM25   : Deletion | hit = 1
DENSE  : Deletion | hit = 1
HYBRID : Deletion | hit = 1

Summary
bm25: hit@1 = 1.00
dense: hit@1 = 0.67
hybrid: hit@1 = 1.00


In [5]:
import json
from pathlib import Path

from lexguard.utils.paths import PROJECT_ROOT
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

benchmark_path = PROJECT_ROOT / "data" / "benchmarks" / "retrieval_smoke_test.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

bm25 = BM25Retriever(chunks)
dense = DenseRetriever(chunks)
hybrid = HybridRetriever(chunks, bm25_weight=0.5, dense_weight=0.5)

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
}

results = []

for item in benchmark:
    query = item["query"]
    expected = item["expected_section"]

    row = {
        "query": query,
        "expected": expected,
    }

    for name, system in systems.items():
        top1 = system.query(query, top_k=1)
        top2 = system.query(query, top_k=2)

        pred_top1 = top1[0].section_title
        pred_top2 = [r.section_title for r in top2]

        row[f"{name}_top1"] = pred_top1
        row[f"{name}_top2"] = pred_top2
        row[f"{name}_hit1"] = int(expected == pred_top1)
        row[f"{name}_hit2"] = int(expected in pred_top2)

    results.append(row)

for row in results:
    print("=" * 80)
    print("QUERY:", row["query"])
    print("EXPECTED:", row["expected"])
    for name in systems:
        print(
            f"{name.upper():7} top1={row[f'{name}_top1']!r} "
            f"| top2={row[f'{name}_top2']} "
            f"| hit@1={row[f'{name}_hit1']} "
            f"| hit@2={row[f'{name}_hit2']}"
        )

print("\nSummary")
for name in systems:
    hit1 = sum(r[f"{name}_hit1"] for r in results) / len(results)
    hit2 = sum(r[f"{name}_hit2"] for r in results) / len(results)
    print(f"{name}: hit@1 = {hit1:.2f}, hit@2 = {hit2:.2f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3817.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6103.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: log retention period
EXPECTED: Data Retention
BM25    top1='Data Retention' | top2=['Data Retention', 'Access Control'] | hit@1=1 | hit@2=1
DENSE   top1='Exceptions' | top2=['Exceptions', 'Data Retention'] | hit@1=0 | hit@2=1
HYBRID  top1='Data Retention' | top2=['Data Retention', 'Access Control'] | hit@1=1 | hit@2=1
QUERY: financial logs retention
EXPECTED: Exceptions
BM25    top1='Exceptions' | top2=['Exceptions', 'Access Control'] | hit@1=1 | hit@2=1
DENSE   top1='Exceptions' | top2=['Exceptions', 'Data Retention'] | hit@1=1 | hit@2=1
HYBRID  top1='Exceptions' | top2=['Exceptions', 'Access Control'] | hit@1=1 | hit@2=1
QUERY: temporary logs deletion
EXPECTED: Deletion
BM25    top1='Deletion' | top2=['Deletion', 'Access Control'] | hit@1=1 | hit@2=1
DENSE   top1='Deletion' | top2=['Deletion', 'Exceptions'] | hit@1=1 | hit@2=1
HYBRID  top1='Deletion' | top2=['Deletion', 'Access Control'] | hit@1=1 | hit@2=1

Summary
bm25: hit@1 = 1.00, hit@2 = 1.00
dense: hit@1 = 0.67, hit@2 =

In [6]:
from pathlib import Path

from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.pdf_loader import load_pdf
from lexguard.ingestion.chunker import build_chunks

cuad_dir = RAW_DATA_DIR / "cuad"
pdf_files = sorted(cuad_dir.glob("*.pdf"))

print("Total PDF files found:", len(pdf_files))
print("First 3 files:")
for f in pdf_files[:3]:
    print("-", f.name)

Total PDF files found: 10
First 3 files:
- AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content%20License%20Agreement.pdf
- ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7_Trademark%20License%20Agreement.pdf
- ArtaraTherapeuticsInc_20200110_8-K_EX-10.5_11943350_EX-10.5_License%20Agreement.pdf


In [7]:
from pathlib import Path

from lexguard.ingestion.pdf_loader import load_pdf

def load_cuad_pdf_folder(folder_path: str, limit: int | None = 5):
    folder = Path(folder_path)
    all_docs = []

    pdf_files = sorted(folder.glob("*.pdf"))
    if limit is not None:
        pdf_files = pdf_files[:limit]

    for file in pdf_files:
        pages = load_pdf(str(file))
        all_docs.append({
            "doc_id": file.stem,
            "title": file.stem,
            "pages": pages,
            "source_path": str(file),
        })

    return all_docs

In [8]:
docs = load_cuad_pdf_folder(str(cuad_dir), limit=5)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

print("Documents loaded:", len(docs))
print("Total chunks:", len(all_chunks))

if all_chunks:
    print("\nFirst chunk example:")
    print("document_id:", all_chunks[0].document_id)
    print("section_title:", all_chunks[0].section_title)
    print("text:", all_chunks[0].chunk_text[:500])

Documents loaded: 5
Total chunks: 100

First chunk example:
document_id: AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content%20License%20Agreement
section_title: None
text: Exhibit 10.19
JOINT CONTENT LICENSE AGREEMENT
This JOINT CONTENT LICENSE AGREEMENT (the “Agreement”), dated February 1, 2018 (the “Effective Date”), is made by and between
WPT Enterprises, Inc., a Delaware corporation, with offices located at 1920 Main Street, Suite 1150, Irvine, CA 92614 (“WPT”), and ZYNGA INC., a
Delaware corporation with offices located at 699 8th Street, San Francisco CA, 94103 (“Zynga US”) and ZYNGA GAME IRELAND LIMITED, a
limited company organized under the laws of Ireland


In [9]:
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.hybrid import HybridRetriever

bm25 = BM25Retriever(all_chunks)
hybrid = HybridRetriever(all_chunks, bm25_weight=0.5, dense_weight=0.5)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3665.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
query = "termination for convenience"

results = hybrid.query(query, top_k=3)

for i, r in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Document:", r.document_title)
    print("Section:", r.section_title)
    print("Text:", r.chunk_text[:800])

Rank: 1
Document: ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7_Trademark%20License%20Agreement
Section: None
Text: 6.3 Waiver. The waiver by one party of a breach or a default of any provision of this Agreement by the other party shall not be construed
as a waiver of any succeeding breach of the same or any other provision, nor shall any delay or omission on the part of a party to exercise or avail
itself of any right, power or privilege that it has or may have hereunder operate as a waiver of any right, power or privilege by such party.
6.4 Waiver of Jury Trial. TO THE FULLEST EXTENT PERMITTED BY APPLICABLE LAW EACH PARTY HEREBY IRREVOCABLY
WAIVES ALL RIGHT OF TRIAL BY JURY IN ANY ACTION, PROCEEDING, CLAIM, OR COUNTERCLAIM ARISING OUT OF OR IN
CONNECTION WITH THIS AGREEMENT OR ANY MATTER ARISING HEREUNDER.
6.5 Notices. Any notice or other communication under this Agreement shall be effective when: (a)
Rank: 2
Document: ArtaraTherapeuticsInc_20200110_8-K_EX-10.5_119